# PROC LOAN을 이용한 학자금 대출 상환 플랜 비교

## 핵심 요약

한 학자금 지원 사무실이 졸업하는 학생 집단이 27,500달러의 대표 잔액을 상환하는 방법을
평가합니다. 연방 고정금리 표준 플랜, 초기 수수료가 있는 민간 재융자, 상한이 있는
변동금리(ARM) 대출, 고용주 지원 바이다운(buydown)이라는 네 가지 상환 구조를
`PROC LOAN`으로 비교합니다. 120개월 기간 동안 네 가지 제안은 견적 시작 금리에서 월
납입액과 총이자가 명확하게 정렬됩니다: 연방 표준 플랜이 이자 총액이 가장 많고(금리
6.53%에서 **10,021.22달러**, 납입액 **312.68달러**), 민간 재융자는 금리는 낮추지만
412.50달러의 수수료가 추가되며(금리 5.99%에서 **9,120.20달러**, 납입액 **305.17달러**),
더 낮은 견적의 ARM(금리 4.75%에서 **7,099.76달러**, 납입액 **288.33달러**)과 고용주
바이다운(금리 4.50%에서 **6,700.67달러**, 납입액 **285.01달러**)이 가장 작은 이자
청구액을 기록합니다. 이어서 `COMPARE` 블록은 36개월, 60개월, 120개월 시점에서 각
플랜의 누적 이자, 누적 원금, 미상환 잔액을 보고하여, 차입자가 재융자하거나 완납할
가능성이 가장 높은 시점에 각 대출이 얼마나 상환되었는지를 보여줍니다.

## 데이터 원본

| 데이터셋 | 행 수 | 설명 | 주요 변수 |
|---------|------|-------------|---------------|
| `borrowers` | 40 | `call streaminit(20260531)`과 `rand('uniform')`으로 인라인 생성한 가상 졸업생 집단의 대출 프로필. 비교를 위한 현실적인 대출 조건을 도출하는 데 사용됩니다. | `student_id` (1001-1040), `balance` (졸업 시점 원금; 이 표본은 11,800달러-47,750달러 범위, 평균 30,771달러), `apr` (연 명목 금리 4.10%-9.10%, 평균 6.50%), `term` (120개월 또는 180개월, 평균 144개월), `origination_pct` (수수료 1.0%-2.0%, 평균 1.50%) |

`PROC LOAN`으로 분석하는 대표 차입자(금액 27,500달러, 120개월 기간, 2026년 7월 시작)는
이 집단의 잔액·금리 분포에서 중간보다 약간 낮은 위치에 있습니다. 외부 데이터나 네트워크
데이터는 사용하지 않습니다. 이 집단은 그럴듯한 대출 조건을 도출하기 위해 존재하며,
정면 비교는 단일 대표 대출에 대해서만 실행됩니다.

# PROC LOAN을 이용한 학자금 대출 상환 플랜 비교

학생들이 졸업하면 학자금 지원 사무실은 경쟁하는 상환 제안들 중에서 선택하도록
도와야 합니다. `PROC LOAN`(SAS/ETS)은 바로 이 작업을 위해 만들어졌습니다: 고정금리,
변동금리(ARM), 바이다운 대출을 상환 계산하고 납입액, 총이자, 상환 진행 상황을
나란히 비교합니다.

이 노트북에서는 다음을 수행합니다:

1. 현실적인 대출 조건을 도출하기 위해 가상의 졸업생 집단을 생성합니다.
2. `PROC MEANS`로 집단을 요약합니다.
3. 대표적인 27,500달러 잔액에 대해 네 가지 상환 플랜을 구성하고 `PROC LOAN`
   (FIXED / ARM / BUYDOWN + COMPARE)으로 비교합니다.
4. 추천된 고정금리 플랜을 단독으로 재실행하여 독립적인 경제성을 확인합니다.

모든 작업은 인라인 가상 데이터로 로컬에서 실행되며, 외부 파일이나 네트워크
접근은 없습니다.

## 1단계 — 가상의 졸업생 집단 생성

40명의 차입자를 시뮬레이션합니다. 각 차입자는 졸업 시점의 원금 잔액, 신용도와 느슨하게
연결된 APR, 표준 상환 기간(10년 또는 15년), 수수료 비율을 뽑습니다. 시드값을 고정하여
데이터를 재현할 수 있습니다.

In [1]:
데이터 borrowers;
   호출 streaminit(20260531);
   길이 plan $ 28;
   반복 student_id = 1001 까지 1040;
      /* 졸업 시점 원금 잔액: 9,500달러 - 48,000달러 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* 신용 등급별 연 명목 APR: 4.0% - 9.5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* 표준 상환 기간: 120개월 또는 180개월 */
      만약 rand('uniform') < 0.6 이면 term = 120;
      아니면 term = 180;
      /* 원금 대비 수수료 비율: 1.0% - 2.0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      출력;
   종료;
실행;



NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## 2단계 — 집단 프로파일링

개별 대출을 모델링하기 전에 잔액, 금리, 기간의 분포를 살펴봅니다. 이를 통해 이후에
이어지는 정면 비교의 기준이 되는 *대표적인* 차입자의 모습을 파악할 수 있습니다.

In [2]:
처리 평균 데이터=borrowers n mean MIN MAX maxdec=2;
   변수 balance apr term origination_pct;
   라벨 balance="대출 잔액($)" apr="연이율(%)" term="상환 기간(개월)" origination_pct="수수료율(%)";
실행;


                                                  The MEANS Procedure

 Variable         Label                        N           Mean     Minimum     Maximum
 --------------------------------------------------------------------------------------
 balance          대출 잔액($)                    40       30771.25    11800.00    47750.00
 apr              연이율(%)                      40           6.50        4.10        9.10
 term             상환 기간(개월)                   40         144.00      120.00      180.00
 origination_pct  수수료율(%)                     40           1.50        1.00        2.00
 --------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3단계 — 대표 차입자를 위한 네 가지 상환 플랜 비교

2026년 7월에 시작하는 120개월(10년) 기간에 걸쳐 대표적인 27,500달러 잔액을 사용하여,
현실적인 네 가지 제안을 정렬합니다:

- **연방 다이렉트 무보조(표준)** — 6.53%의 단순 고정금리 대출.
- **민간 재융자(수수료 포함)** — 5.99%로 더 낮은 고정금리이지만 412.50달러의
  초기 비용(`INIT=`)이 있습니다.
- **민간 변동금리 ARM** — 조정당 1% / 생애 5%의 `CAPS=`, 9.75%의 `MAXRATE=`,
  연 단위 `ADJUSTFREQ=`, 그리고 `WORSTCASE` 키워드를 갖는 4.75% 변동금리 대출.
- **고용주 2-1 바이다운** — 견적 일정상 `BDRATES=`를 통해 2년 차에 5.50%,
  이후 6.50%로 단계적으로 오르는 보조금 지원 4.50% 시작 금리.

`COMPARE` 문은 22%의 `TAXRATE=`와 4%의 최소 매력 수익률(`MARR=`)로 36개월, 60개월,
120개월 시점의 대출 간 비교 뷰를 요청합니다. `OUTSUM=`과 `OUTCOMP=`는 대출별 요약과
비교 행을 캡처합니다. 아래 리스팅은 각 플랜과 각 체크포인트에 대해 **누적 지급 이자,
누적 상환 원금, 미상환 잔액**을 보고하여, 후보 대출들 간의 명확한 상환 진행 상황
뷰를 제공합니다.

> **금리 조정에 대한 참고 사항.** Jenner의 `PROC LOAN`은 모든 플랜을 견적된
> **시작** 금리로 상환 계산하므로, ARM의 `CAPS`/`MAXRATE`/`WORSTCASE` 설정과
> 바이다운의 `BDRATES` 단계는 리스팅에 대출 조건으로 표시되지만 납입액 계산에는
> **적용되지 않습니다** — 아래의 ARM과 바이다운 수치는 전체 기간 동안 4.75%와
> 4.50% 시작 금리를 그대로 유지한 것을 반영합니다. 이 두 총액은 최악의 경우
> 상한이 아니라 최선의 경우(시작 금리) 비용으로 취급하십시오.

In [3]:
처리 loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         라벨="연방 표준(무보조)";

   fixed rate=5.99 init=412.50
         라벨="민간 재융자(수수료)";

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       라벨="변동금리 ARM(최악)";

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           라벨="고용주 바이다운";

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
실행;


                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: 연방 표준(무보조)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: 민간 재융자(수수료)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: 변동금리 ARM(최악)
    Loan Type:                   Adjustable Rate
    Amount (Principal):                27500.00
    Interest Rate (annual %):            4.7500


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## 4단계 — 추천된 고정금리 플랜 단독 재실행

납입액의 확실성을 중시하는 차입자에게는 연방 고정금리 표준 플랜이 안전한 기본
선택입니다. 이를 단독으로 재실행하여, 네 가지 비교에서 확인된 것과 동일한
월 납입액 **312.68달러**, 총 지급액 **37,521.22달러**, 총이자 **10,021.22달러**의
핵심 경제성을 독립적인 대출 요약으로 다시 확인합니다.

In [4]:
처리 loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed 라벨="연방 표준(무보조)";
실행;


                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: 연방 표준(무보조)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: 연방 표준(무보조)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4         21035.90        3752.12        1301.15        2450.97       


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## 결과 해석

네 가지 플랜 모두 120개월에 걸쳐 동일한 27,500달러 원금을 완전히 상환합니다
(COMPARE 표에서 모든 플랜의 미상환 잔액이 120개월 시점에 0.00달러에 도달합니다).
따라서 결정은 **월 납입액**과 **견적 금리에서의 총이자**에 달려 있습니다:

| 플랜 | 금리 | 납입액 | 총이자 | 비고 |
|------|------|---------|----------------|-------|
| 연방 다이렉트 표준 | 6.53% | 312.68달러 | 10,021.22달러 | 수수료 없음; 가장 강력한 차입자 보호 |
| 민간 재융자 | 5.99% | 305.17달러 | 9,120.20달러 | 412.50달러 초기 수수료 |
| 민간 변동금리 ARM | 4.75%(시작) | 288.33달러 | 7,099.76달러 | 금리가 오를 수 있음; 수치는 시작 금리 기준 비용 |
| 고용주 2-1 바이다운 | 4.50%(시작) | 285.01달러 | 6,700.67달러 | 고용 지속 여부에 좌우됨 |

- **연방 표준** 플랜은 이자 기준으로 가장 비싸지만(10,021.22달러) 수수료 없이
  고정되고 예측 가능한 312.68달러 납입액을 제공합니다.
- **민간 재융자**는 금리를 5.99%로 낮춰(9,120.20달러 이자, 연방 플랜보다
  901달러 적음) 412.50달러의 초기 수수료를 부과하므로, 연방 플랜 대비 순이익은
  이자 절감분에서 수수료를 뺀 약 488달러 정도이며, 이는 재융자되지 않고
  대출이 충분히 오래 유지될 때만 의미가 있습니다.
- **ARM**과 **바이다운**은 여기서 가장 낮은 이자를 보이는데(7,099.76달러와
  6,700.67달러), 이는 순전히 **시작** 금리가 고정금리 제안보다 훨씬 낮기
  때문입니다. 3단계에서 언급했듯이 Jenner는 이 시작 금리를 그대로 유지하므로
  이 수치들은 최선의 경우 값입니다: 실제로 상승하는 ARM이나 고용주 보조를
  잃는 바이다운은 더 높아질 것이며, 납입액도 보장되지 않습니다.

`COMPARE` 표는 각 플랜이 얼마나 빠르게 상환되는지를 보여줌으로써 이를 더욱
명확히 합니다. 36개월 시점에 연방 플랜은 4,792.27달러의 이자를 지급하고
6,464.10달러의 원금을 상환했지만(잔액 21,035.90달러), 바이다운은 이자
3,263.97달러만 지급하고 6,996.24달러의 원금을 상환했습니다(잔액
20,503.76달러) — 금리가 낮은 플랜은 첫 3년 동안 이자가 적게 들면서도 원금을
더 빨리 갚아 나갑니다. 위험 회피 성향의 졸업생에게는 연방 표준 플랜의 금리
확실성이 더 높은 이자를 정당화하는 경우가 많지만, 안정적이고 지속적인 고용에
자신 있는 차입자에게는 실제로 금리를 고정하는 옵션 중에서 바이다운의 보조
시작 금리가 가장 큰 절감 효과를 제공합니다.